# Knowledge base (ChromaDB)

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch
import pandas as pd
import chromadb
from tqdm.notebook import tqdm
from IPython.display import clear_output

CSV_PATH = "./knowledge/clean_youtube_watch_log.csv"
THUMBNAIL_PATH = "./knowledge/thumbnails"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def embed_text(text: str):
    """Embed a text using CLIP"""
    inputs = proc(
        text=[text], return_tensors="pt", truncation=True, padding=True
    )  # WARNING! Truncation may lead to loss of information for long texts
    with torch.no_grad():
        emb = model.get_text_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True)  # Normalize

def embed_image(image: Image.Image):
    """Embed an image using CLIP"""
    inputs = proc(images=image, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True)  # Normalize

In [ ]:
def build_index():
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.create_collection(name="youtube_videos")

    # Load video metadata and thumbnails
    df = pd.read_csv(CSV_PATH)
    video_ids = list(dict.fromkeys(df['video_id']))       # ignore duplicates
    titles = [df[df['video_id'] == vid]['video_title'].iloc[0] for vid in video_ids]
    thumbnails = [Image.open(f'{THUMBNAIL_PATH}/{vid}.jpg') for vid in video_ids]
    
    # Embed all texts and images
    vecs = []
    for t, img in tqdm(zip(titles, thumbnails), total=len(titles), desc="Embedding Image / Text pairs"):
        text_emb = embed_text(t)
        img_emb  = embed_image(img)
        # fusion: 70% text, 30% image
        vecs.append(0.7 * text_emb + 0.3 * img_emb)

    # Convert to numpy array
    vecs = torch.cat(vecs).cpu().numpy()

    # Add embeddings to the collection
    collection.add(
        ids=video_ids,            # identifier for each entry (e.g., video ID)
        embeddings=vecs,    # list of embeddings (fused text + image)
        documents=titles    # optional: store the original titles as metadata
    )

    return collection

def load_index():
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.get_collection(name="youtube_videos")
    return collection

In [ ]:
# client = chromadb.PersistentClient(path="./chroma_db")
# client.delete_collection(name="youtube_videos")

# build_index()

# load_index()

Embedding Image / Text pairs:   0%|          | 0/897 [00:00<?, ?it/s]

Collection(name=youtube_videos)

# Building the Agent (LangChain)

In [ ]:
from dataclasses import dataclass
from typing import Annotated, TypedDict
import pandas as pd
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from dataclasses import asdict
import ipywidgets as widgets
from PIL import Image
from IPython.display import display
from IPython.display import clear_output
import html
import ipywidgets as widgets
from IPython.display import display
from dataclasses import dataclass
from typing import Annotated, TypedDict
import pandas as pd
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from dataclasses import asdict
import ipywidgets as widgets
from PIL import Image
from IPython.display import display
from IPython.display import clear_output
import html
import ipywidgets as widgets
from IPython.display import display

llm = ChatOllama(
    model="qwen3.5:2b",
    temperature=0
)

## 1. Data types

In [13]:
@dataclass
class WatchItem:
    user_id: str
    video_id: str
    video_title: str
    timestamp: str      # ISO format

    @property
    def thumbnail(self) -> Image.Image:
        return Image.open(f'./knowledge/thumbnails/{self.video_id}.jpg')

## 2. Tools

In [14]:
@tool
def retrieve_session(user_id: str, sorted = True, n: int = None) -> list[WatchItem]:
    """
    Loads a user's watch session, sorted by the most recent first. 
    (Optionally) limited to the top N results.
    """
    df = pd.read_csv(CSV_PATH)
    df = df[df['user_id'] == user_id]   # Filter by user ID

    # Sort and limit
    if sorted:
        df = df.sort_values("watch_date", ascending=False)
    if n is not None:
        df = df.head(n)

    # Map to dataclass
    return df.apply(
        lambda row: asdict(WatchItem(
            user_id=row['user_id'],
            video_id=row['video_id'],
            video_title=row['video_title'],
            timestamp=row['watch_date']
        )), axis=1).tolist()

TOOLS = [retrieve_session]

## 3. Building the agent

In [15]:
# Bind the tools
llm = llm.bind_tools([retrieve_session])

# State definition for LangGraph
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


# ---------- Node 1: LLM agent  ----------
SYSTEM_PROMPT = """
You are a YouTube watch-history analysis agent for rabbit hole prevention.

You help answer questions about a user's history.

Rules:
- If you need the list of videos a user watched, call retrieve_session.
- You can limit the number of videos retrieved if you want, but by default retrieve the whole session.
- When you know the answer, respond directly to the user in one short sentence.
"""

def agent_node(state: AgentState):
    """
    When LLM node is invoked, it receives the System Prompt + the conversation history. 
    After reasoning, it creates a new message with either an answer or a tool call.
    """
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


# ---------- Node 2: Tool router ----------
def route_tools(state: AgentState):
    """
    When router node is invoked, it checks the last message (from LLM node) for tool calls. 
    If there are tool calls, it routes to the tools node, otherwise it ends the graph execution.
    """
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END


# ---------- Graph ----------
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(TOOLS))      # pre-built tool node

# Add edges
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", route_tools, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

graph = builder.compile()

# 4. Testing the agent

In [ ]:
user_id = "25"
print("Agent 🤖 is ready. Type 'exit' to quit.\n")

while True:
    user = input("\nYou: ").strip()
    if user == '' or user.lower() == "exit":
        break

    prompt = (
        f"The user_id is {user_id}. "
        f"Answer to this question about that user's session: {user}"
    )

    stream = graph.stream(
        {"messages": [HumanMessage(content=prompt)]},
        stream_mode="messages",
        version="v2",
    )

    print("\n\nYou 🙋‍♂️:", user)
    print("\nAgent 🪨: ... ")

    for part in stream:
        content = (
            part["data"][0].content
            if "data" in part
            and isinstance(part["data"], tuple)
            and hasattr(part["data"][0], "content")
            else None
        )

        if content:
            print(content, end="")

print("\n\nAgent 🪨: Okay then you don't like me. Goodbye.")

Agent 🤖 is ready. Type 'exit' to quit.



You 🙋‍♂️: what can you do?

Agent 🪨: ... 
I was able to retrieve your watch session for user_id 25, but it appears there are no videos recorded in your history.

You 🙋‍♂️: okay

Agent 🪨: ... 
I'm sorry, but I couldn't find any relevant information in your watch history for that user ID.